# What changed in the room? Run on Colab

For machines without a suitable GPU. Runtime > Change runtime type > **T4 GPU**.

1. Zip your whole `roomdiff` folder, with the videos in `data/corridor/` (`before_Left.MOV`, `before_Right.MOV`, `after_Left.MOV`, `after_Right.MOV`), as `roomdiff.zip`.
2. Put it in a Google Drive folder called `roomdiff_data`.
3. Run all cells. Results are copied to `roomdiff_data/results/` in your Drive.

For your own videos, edit `PAIRS` below. HDR iPhone videos are tone-mapped automatically (Colab has ffmpeg).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/roomdiff_data'
!ls -la "$DATA"

In [ ]:
!rm -rf /content/roomdiff && unzip -q "$DATA/roomdiff.zip" -d /content/
%cd /content/roomdiff
!pip -q install -r requirements.txt
!test -d third_party/vggt || (git clone -q https://github.com/facebookresearch/vggt.git third_party/vggt && git -C third_party/vggt checkout -q a288dd0f14786c93483e45524328726ab7b1b4ce)
!pip -q install -e third_party/vggt
!python -m roomdiff doctor

## Settings

In [ ]:
PAIRS = {   # name: (before, after); paths may omit the video extension
    'corridor_right': ('data/corridor/before_Right', 'data/corridor/after_Right'),
    'corridor_same':  ('data/corridor/before_Left',  'data/corridor/after_Left'),
    'corridor_cross': ('data/corridor/before_Left',  'data/corridor/after_Right'),
    'corridor_null':  ('data/corridor/before_Left',  'data/corridor/before_Right'),  # nothing changed
}
HEIGHT = 1.20        # phone height above the floor in metres

## Run every comparison

`corridor_null` compares two recordings with nothing changed: anything it reports is a false positive.

In [ ]:
import subprocess
for name, (b, a) in PAIRS.items():
    print('=====', name)
    subprocess.run(['python', '-m', 'roomdiff', 'run', '--before', b, '--after', a,
                    '--out', f'results/{name}', '--camera-height', str(HEIGHT)], check=True)

## Look and save

In [ ]:
from IPython.display import Image, display
for r in PAIRS:
    print(open(f'results/{r}/report.md').read())
    display(Image(f'results/{r}/topdown.png', width=650))
!mkdir -p "$DATA/results" && cp -r results/* "$DATA/results/"
!ls -la "$DATA/results"